# Extending a single state anchor via a two-sided random walk

This notebook covers the simplest case: each channel has **one** disclosed time point. `construct_states_prior`
discloses the latent state at that anchor and leaves every other step at `P_obs = inf` (undisclosed). For a single
channel touched by no other data, those null steps are **not** truly uninformed: the driftless random-walk
transition already implies an exact marginal that propagates the anchor forward *and* backward,

$$a_{\text{obs}}[t] = a^{*}, \qquad P_{\text{obs}}[t] = P^{*} + |t - t^{*}|\,Q,$$

where $(a^{*}, P^{*})$ is the disclosed anchor and $Q = \sigma_q^2$ is the per-state process variance. The mean is
carried flat; the variance grows linearly with the lag, forming a "cone" that pinches to $P^{*}$ at the anchor.

`extend_states_prior_nearest(ssp_priors, Q)` fills the `inf` entries with this marginal. It is the exact marginal
only for an **isolated** channel, so treat the output as a prior fed into the augmented-measurement step — not a
final posterior. States with no anchor stay `inf`, and $Q \to \infty$ recovers the un-extended prior (only the
anchor informed).

Section 4 shows that with one anchor, every way of computing this band gives the same answer. With several anchors
they part ways, which is the subject of
[`ssp_08_multi_anchor_prior_extension.ipynb`](./ssp_08_multi_anchor_prior_extension.ipynb).

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)  # float64 for a clean diffuse-prior smoother

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from jax import numpy as jnp

from bunobee.models.ssp.plotting import plot_prior_heatmap, plot_states
from bunobee.models.ssp.prior import disclosed_idx, extend_states_prior_nearest, extend_states_prior_smoothed
from bunobee.simulation.ssp import construct_states_prior

## 1. Construct the disclosed states prior

We synthesise a weekly (52-step) prior over a level plus three media regressors, disclosing the ground-truth
latent state at a **single time point**. The level carries no finite-variance anchor (its variance stays `inf`),
which later demonstrates the "no anchor -> stays undisclosed" behaviour.

In [ ]:
regressors = ["tv", "search", "social"]
state_labels = ["level", *regressors]
n_states = len(state_labels)
n_steps = 52

true_states = jnp.array([0.0, 0.8, 0.5, 0.3])  # level ignored (var stays inf)

states_prior = construct_states_prior(
    n_steps=n_steps,
    n_states=n_states,
    true_states=true_states,
    regressors=regressors,
    n_periods=1,
    n_points=1,
    seed=1,
    obs_scale=0.1,
)

dates = pd.date_range("2024-01-01", periods=n_steps, freq="W")
print("disclosed anchor step:", disclosed_idx(states_prior))
print("finite fraction (base):", np.isfinite(states_prior["P_obs"].values).mean().round(3))

In [ ]:
plot_prior_heatmap(states_prior, quantity="both")
plt.show()

## 2. Extend the prior along the random walk

`extend_states_prior_nearest` fills every undisclosed step of each anchored state with the random-walk marginal
around its anchor. Here we use a single scalar process variance `Q`; a length-`n_states` array can instead give
each state its own slope.

In [ ]:
# Extend the disclosed states prior and visualise the result
Q = 0.02  # per-state process variance sigma_q**2 (scalar broadcast to every state)
ext_states_prior = extend_states_prior_nearest(states_prior, Q)

print("finite fraction (extended):", np.isfinite(ext_states_prior["P_obs"].values).mean().round(3))
print("level stays fully undisclosed:", bool(np.all(np.isinf(ext_states_prior["P_obs"].values[:, 0]))))

# Plot the extended time-point prior as a states x time heatmap
plot_prior_heatmap(ext_states_prior, quantity="both")
plt.show()

In [ ]:
# Sanity check: an undisclosed step matches P* + |t - t*|·Q for the anchor.
tv = state_labels.index("tv")
t_star = int(disclosed_idx(states_prior)[0])
for t in (10, 20, 40):
    expected = states_prior["P_obs"].values[t_star, tv] + abs(t - t_star) * Q
    got = ext_states_prior["P_obs"].values[t, tv]
    print(f"tv step t={t:>2}: anchor t*={t_star}, P_obs={got:.3f} (expected {expected:.3f})")

ext_states_prior

## 3. Contrast the prior with vs. without the extension

We draw pseudo-samples from each prior's per-step marginal $\mathcal{N}(a_{\text{obs}}, P_{\text{obs}})$
and hand both to `plot_states`, overlaying them in a single figure. Undisclosed steps (`inf` variance)
become `NaN`, so the **anchors-only** prior has no ribbon, only the red marker at its one disclosed step, while the
**RW-extension** ribbon is a continuous cone that narrows to the anchor and widens with lag. The level panel stays
empty (no anchor to extend).

In [ ]:
def prior_samples(ssp_priors, n_draws=2000, seed=0):
    """Draw samples from a prior's per-step Gaussian marginal.

    Parameters
    ----------
    ssp_priors : xr.Dataset
        Prior with ``a_obs`` / ``P_obs`` over ``(time, state)``.
    n_draws : int, optional
        Number of Monte Carlo draws, by default 2000.
    seed : int, optional
        RNG seed, by default 0.

    Returns
    -------
    np.ndarray, shape (n_draws, n_steps, n_states)
        Draws with undisclosed (``inf``-variance) steps set to ``NaN`` so the
        ribbon shows a gap there.
    """
    rng = np.random.default_rng(seed)
    a = np.asarray(ssp_priors["a_obs"].values, dtype=float)
    p = np.asarray(ssp_priors["P_obs"].values, dtype=float)
    draws = a[None] + np.sqrt(p)[None] * rng.standard_normal((n_draws, *a.shape))
    draws[:, ~np.isfinite(p)] = np.nan
    return draws


posterior = {
    "anchors only": prior_samples(states_prior),
    "RW extension": prior_samples(ext_states_prior),
}

fig, axes = plot_states(
    posterior,
    dates.values,
    state_labels,
    states_key=["anchors only", "RW extension"],
    obs_idx=disclosed_idx(states_prior),
    a_obs=states_prior["a_obs"].values,
    P_obs=states_prior["P_obs"].values,
    title=f"Two-sided random-walk extension of state anchors (Q={Q})",
    n_cols=2,
    colors={"anchors only": "darkgreen", "RW extension": "steelblue"},
)
plt.show()

## 4. With one anchor, all three methods agree

The same `tv` channel drawn three ways, each as a 90% band on a shared y-axis:

1. **Sampling** -- the recipe from section 3: draw Monte Carlo samples from the extended prior's marginal and take
   the 5% / 95% quantiles.
2. **Nearest-anchor heuristic** -- the same extended prior, with the band read directly from
   $a_{\text{obs}} \pm 1.645\,\sqrt{P_{\text{obs}}}$.
3. **KF + RTS smoother** -- `extend_states_prior_smoothed`, which runs a forward Kalman filter and a backward RTS
   smoother over the same random walk, with the anchor fed in as an observation.

With one anchor there is nothing to fuse. The forward pass carries the anchor ahead, the backward pass carries it
back, and neither has a second source of information to weigh it against, so the smoother reduces to
$P^{*} + |t - t^{*}|\,Q$.

In [ ]:
def exact_band(ssp_prior, s, z=1.645):
    """Return the exact central 90% band of one state's per-step Gaussian marginal.

    Parameters
    ----------
    ssp_prior : xr.Dataset
        Prior with ``a_obs`` / ``P_obs`` over ``(time, state)``.
    s : int
        Index of the state to read.
    z : float, optional
        Standard-normal quantile setting the band width, by default 1.645 (90%).

    Returns
    -------
    tuple of np.ndarray
        ``(lower, centre, upper)``, each of shape ``(n_steps,)``.
    """
    mean = ssp_prior["a_obs"].values[:, s]
    half_width = z * np.sqrt(ssp_prior["P_obs"].values[:, s])
    return mean - half_width, mean, mean + half_width


smoothed_states_prior = extend_states_prior_smoothed(states_prior, Q)

steps = np.arange(n_steps)
sampled_lo, sampled_mid, sampled_hi = np.quantile(posterior["RW extension"][:, :, tv], [0.05, 0.5, 0.95], axis=0)
bands = [
    ("(i) sampling", (sampled_lo, sampled_mid, sampled_hi), "steelblue"),
    ("(ii) nearest-anchor heuristic, exact", exact_band(ext_states_prior, tv), "steelblue"),
    ("(iii) KF + RTS smoother, exact", exact_band(smoothed_states_prior, tv), "darkorange"),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for ax, (name, (lo, mid, hi), color) in zip(axes, bands):
    ax.fill_between(steps, lo, hi, color=color, alpha=0.25, linewidth=0, label="90% band")
    ax.plot(steps, mid, color=color, lw=2, label="centre")
    ax.scatter(
        t_star,
        states_prior["a_obs"].values[t_star, tv],
        color="crimson",
        marker="x",
        s=50,
        zorder=3,
        label="raw anchor",
    )
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("t")
    ax.grid(alpha=0.2)
axes[0].set_ylabel("tv state value")
axes[0].legend(fontsize=8, loc="upper left")
fig.suptitle(f"90% prior band of a single-anchor channel under each method (Q={Q})", fontsize=11)
plt.tight_layout()
plt.show()

Panels (ii) and (iii) are the same band, and (i) differs from them only by Monte Carlo noise. A direct numeric
check across the whole series, for every state:

In [ ]:
for i, label in enumerate(state_labels):
    nearest_a, nearest_p = ext_states_prior["a_obs"].values[:, i], ext_states_prior["P_obs"].values[:, i]
    smooth_a, smooth_p = smoothed_states_prior["a_obs"].values[:, i], smoothed_states_prior["P_obs"].values[:, i]
    if np.all(np.isinf(nearest_p)) and np.all(np.isinf(smooth_p)):
        print(f"{label:>8}: no anchor, P_obs stays inf under both")
        continue
    d_mean = np.abs(nearest_a - smooth_a).max()
    d_var = np.abs(nearest_p - smooth_p).max()
    print(f"{label:>8}: max|delta mean| = {d_mean:.2e}   max|delta variance| = {d_var:.2e}")

## Takeaways

- The **RW extension** turns a single anchor into a full time-point prior: the mean is held flat and the variance
  opens into a symmetric cone $P^{*} + |t - t^{*}|\,Q$ around it. A state with **no** anchor (here the level) keeps
  `P_obs = inf`.
- **With one anchor, the methods are equal.** Sampling, the nearest-anchor formula and the KF + RTS smoother all give
  the same prior: the two exact methods differ only by floating-point noise, and sampling by Monte Carlo noise. For
  this case the simple formula is all you need.
- Increasing `Q` widens the cone faster; $Q \to \infty$ collapses back to the anchor-only prior. Feed the extended
  prior into the augmented-measurement step for a quick de-biased read on an isolated channel.
- With **several** anchors this equality breaks:
  [`ssp_08_multi_anchor_prior_extension.ipynb`](./ssp_08_multi_anchor_prior_extension.ipynb) shows why.